# 📰 Fake News Detection Using Feature-Based Machine Learning Models

This notebook explores fake news classification using fine-tuning on an LLM (specifically - LLaMA 3.1 8B instruct).


### 🎯 Objective
To build and evaluate classical ML models (e.g., Logistic Regression, Random Forest, etc.) for the task of fake news detection, based on interpretable features. This will serve as a baseline before comparing with more complex models such as those based on LLMs.

---

*Let’s dive into data preprocessing, feature selection, and model evaluation!*

# 1. Set-up

## 1.1. Imports & Globals

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)
import torch
import os
from peft import get_peft_model, LoraConfig, TaskType
from huggingface_hub import login

In [ ]:
# Set up enviroment and global variables
LLM_MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
api_key = 'hf-api-key'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

MAX_LENGTH = 1024
scaler = StandardScaler()


DATA_PATH = './data'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

login(api_key)

df = pd.read_csv(os.path.join(DATA_PATH, 'cleaned_news_dataset_w_derived_columns.csv'))
df['text'] = df['text'].str.replace(r'^[A-Z ,./\-a-z]\s\(Reuters\)\s*[-–—]\s', '', regex=True)
df["label"] = df["label"].astype(int)

In [6]:
def sample_balanced_news(df, sample_size=300):
    # Keep only the relevant columns
    df = df[['title', 'text', 'label']]

    # Check that both classes exist
    labels = df['label'].unique()
    if len(labels) < 2:
        raise ValueError("The dataset must contain at least two label classes.")

    # Calculate per-class sample size
    per_class = sample_size // 2

    # Sample per class
    df_0 = df[df['label'] == 0].sample(n=per_class, random_state=42)
    df_1 = df[df['label'] == 1].sample(n=per_class, random_state=42)

    # Concatenate and shuffle
    balanced_sample = pd.concat([df_0, df_1]).sample(frac=1, random_state=42).reset_index(drop=True)

    return balanced_sample


df = sample_balanced_news(df)

## 1.2. Custom Functions

The following sections will set up the functions used for the training process

### LLM Loading

In [7]:
def load_llm_model(model_name=LLM_MODEL_NAME, api_key= api_key):
    tokenizer = AutoTokenizer.from_pretrained(model_name, token=api_key, device_map="cpu")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        token=api_key,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    )

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=["q_proj", "v_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM
    )

    model = get_peft_model(model, lora_config)
    model.train()
    model.to(DEVICE)

    return tokenizer, model

def print_trainable_params(model):
    trainable_params = 0
    all_params = 0
    for name, param in model.named_parameters():
        all_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
            print(f"Trainable: {name}, shape: {param.shape}")
    print(f"Trainable params: {trainable_params} / {all_params} ({100 * trainable_params / all_params:.2f}%)")

### Data Prep Functions for Training

In [8]:
def prepare_text_input(df):
    df = df.copy()
    df['title'] = df['title'].fillna('')
    df['text'] = df['text'].fillna('')

    df['article_content'] = df.apply(
        lambda row: f"Title: {row['title']}\n\nText: {row['text']}" if row['title'].strip() else row['text'],
        axis=1
    )

    label_map = {1: "Fake", 0: "Real"}
    df['response'] = df['label'].map(label_map)

    df['formatted_text'] = df.apply(lambda row:
        f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        f"You are a news article classifier. Analyze news articles and classify them as either 'Fake' or 'Real'. Provide only the classification without explanation.<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>\n\n"
        f"Please classify this news article:\n\n{row['article_content']}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{row['response']}<|eot_id|>", axis=1)

    return df[['formatted_text', 'response', 'article_content']]

def prepare_data_splits(df, val_size=0.2, test_size=0.02, random_state=42):
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        stratify=df['label'],
        random_state=random_state
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        stratify=train_val_df['label'],
        random_state=random_state
    )

    print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
    return train_df, val_df, test_df

### LLM Training Helpers

In [9]:
def tokenize_function(example, tokenizer, max_length=MAX_LENGTH):
    tokenized = tokenizer(
        example["formatted_text"],
        truncation=True,
        padding="max_length",
        max_length=max_length,
        return_tensors="pt"
    )

    # For causal LM, labels = input_ids with padding masked to -100
    input_ids = tokenized["input_ids"]
    labels = input_ids.clone()
    labels[input_ids == tokenizer.pad_token_id] = -100

    return {
        "input_ids": input_ids.squeeze(0),
        "attention_mask": tokenized["attention_mask"].squeeze(0),
        "labels": labels.squeeze(0),
    }


def create_llm_datasets(train_df, val_df, test_df, tokenizer, max_length=1024):
    train_prepared = prepare_text_input(train_df)
    val_prepared = prepare_text_input(val_df)
    test_prepared = prepare_text_input(test_df)

    train_ds = Dataset.from_pandas(train_prepared)
    val_ds = Dataset.from_pandas(val_prepared)
    test_ds = Dataset.from_pandas(test_prepared)

    train_ds = train_ds.map(
        lambda x: tokenize_function(x, tokenizer, max_length),
        batched=False,
        remove_columns=train_ds.column_names
    )
    val_ds = val_ds.map(
        lambda x: tokenize_function(x, tokenizer, max_length),
        batched=False,
        remove_columns=val_ds.column_names
    )
    test_ds = test_ds.map(
        lambda x: tokenize_function(x, tokenizer, max_length),
        batched=False,
        remove_columns=test_ds.column_names
    )

    train_ds.set_format(type="torch")
    val_ds.set_format(type="torch")
    test_ds.set_format(type="torch")

    return train_ds, val_ds, test_ds

### Train & Evaluation

In [10]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling, EarlyStoppingCallback, TrainerCallback
import torch

def train_llm_model(model, tokenizer, train_ds, val_ds, output_dir="./llama_fake_news_classifier_gen"):
    def clear_cuda_memory():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

    class ClearMemoryCallback(TrainerCallback):
        def on_step_end(self, args, state, control, **kwargs):
            if state.global_step % 10 == 0:
                clear_cuda_memory()

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False  # because this is causal LM (not masked LM)
    )

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="steps",
        eval_steps=500,
        save_steps=1000,
        logging_steps=100,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=2,
        num_train_epochs=10,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=100,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        remove_unused_columns=False,
        fp16=False,
        report_to=None,
    )

    data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3), ClearMemoryCallback()]
    )

    trainer.train()
    trainer.save_model()
    tokenizer.save_pretrained("./llama_fake_news_classifier_gen")


# 2. Train

The following section will train the models to classify news as true\false

In [11]:
# set up the data splits and prepare the datasets
train_df, val_df, test_df = prepare_data_splits(df)

Train: 235, Val: 59, Test: 6


In [ ]:
torch.cuda.empty_cache()

# load the LLM model and tokenizer
tokenizer, model = load_llm_model()

In [ ]:
# tokenize and prepare datasets for LLM
train_ds, val_ds, test_ds = create_llm_datasets(train_df, val_df, test_df, tokenizer)

In [ ]:
# train the LLM model
trainer = train_llm_model(model, tokenizer, train_ds=train_ds, val_ds=val_ds)

# 3. Evaluation

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, confusion_matrix
import numpy as np

def evaluate(model, tokenizer, dataset, device, max_new_tokens=5):
    model.eval()
    all_preds = []
    all_labels = []

    for example in dataset:
        article_content = example['article_content']
        true_label = example['response']  # "Fake" or "Real"

        prompt = (
            "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            "You are a news article classifier. Analyze news articles and classify them as either 'Fake' or 'Real'. Provide only the classification without explanation.<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            f"Please classify this news article:\n\n{article_content}<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
        pred_text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip().lower()

        if "fake" in pred_text:
            pred_label = 1
        elif "real" in pred_text:
            pred_label = 0
        else:
            print(f"⚠️ Unrecognized output: {pred_text}")
            continue

        true_label_id = 1 if true_label.lower() == "fake" else 0

        all_preds.append(pred_label)
        all_labels.append(true_label_id)

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds)
    recall = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
    fpr = fp / (fp + tn)
    coverage = len(all_preds) / len(dataset)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "fpr": fpr,
        "coverage": coverage,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    }


In [ ]:
test_df = pd.read_csv(os.path.join(DATA_PATH, 'filtered_test_data.csv'))
test_df = prepare_text_input(test_df)
eval_data = test_df.to_dict(orient='records')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from peft import PeftModel
model = PeftModel.from_pretrained(model, "./llama_fake_news_classifier").to(device)

metrics = evaluate(model, tokenizer, eval_data, device)
print(metrics)

/home/omertole/.conda/envs/big_data_env/lib/python3.9/site-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/home/omertole/.conda/envs/big_data_env/lib/python3.9/site-packages/peft/peft_model.py:569: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.1.self_attn.q_proj.lora_B.default.weight', 'base_model

{'accuracy': 0.4440357330530741, 'precision': 0.4440357330530741, 'recall': 1.0, 'f1': 0.6149927219796215, 'fpr': np.float64(1.0), 'coverage': 1.0, 'tp': np.int64(845), 'fp': np.int64(1058), 'tn': np.int64(0), 'fn': np.int64(0)}
